In [ ]:
!git clone -b lab https://github.com/keizandrea/deep-alg-pro.git
%cd deep-alg-pro-lab
!pip install biopython torchmetrics

Cloning into 'deep-alg-pro'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 152 (delta 63), reused 152 (delta 63), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 2.70 MiB | 6.53 MiB/s, done.
Resolving deltas: 100% (63/63), done.
[Errno 2] No such file or directory: 'deep-alg-pro-lab'
/content
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 11.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 16.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torchmetrics]


In [ ]:
%cd /content/deep-alg-pro
!git pull origin lab

/content/deep-alg-pro
From https://github.com/keizandrea/deep-alg-pro
 * branch            lab        -> FETCH_HEAD
Already up to date.


In [ ]:
!git log --oneline -3
!grep -n "task='binary'" Train.py

82a9b19 (HEAD -> lab, origin/lab, origin/HEAD) Ablation models, --model flag, torchmetrics fix, droput fix
061732a Add ablation models & model selection
bcd40a9 Update all.train.fasta
68:    accuracy = torchmetrics.Accuracy(task='binary').to(device)
69:    recall = torchmetrics.Recall(task='binary').to(device)
70:    precision = torchmetrics.Precision(task='binary').to(device)
71:    auroc = torchmetrics.AUROC(task='binary').to(device)
72:    f1 = torchmetrics.F1Score(task='binary').to(device)


In [ ]:
!nvidia-smi -L
!pip install -q biopython torchmetrics

GPU 0: Tesla T4 (UUID: GPU-26d114fa-6093-9916-aaa6-3b6e18bbe6a8)


In [ ]:
%%time
!python main.py -i data/all.train.fasta --epochs 1 -b 72 --mode train --model convNet | tail -n 3


[k: 1] valid_loss: 4.246 accuracy_value: 0.668651 recall_value: 0.682731 precision_value: 0.658915 auroc_value: 0.737964 f1_value: 0.670611
valid_loss: 0.425 accuracy_value: 0.066865 recall_value: 0.068273 precision_value: 0.065891 auroc_value: 0.073796 f1_value: 0.067061
Thu Oct 29 12:36:12 2026
CPU times: user 67.7 ms, sys: 9.91 ms, total: 77.6 ms
Wall time: 39.1 s


In [ ]:
%%time
!python main.py -i data/all.train.fasta --epochs 1 -b 72 --mode train --model ATTnet | tail -n 3


In [ ]:
!rm -f convNet.1.* ATTnet.1.* valid.log train.log


In [ ]:
!python main.py -i data/all.train.fasta --epochs 120 --lr 0.0001 -b 72 --mode train --model convNet | tail -n 3


[k: 1] valid_loss: 2.029 accuracy_value: 0.873016 recall_value: 0.930612 precision_value: 0.829091 auroc_value: 0.955622 f1_value: 0.876923
valid_loss: 0.203 accuracy_value: 0.087302 recall_value: 0.093061 precision_value: 0.082909 auroc_value: 0.095562 f1_value: 0.087692
Thu Oct 29 12:37:49 2026


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/deepalgpro_ablation
!cp *.1.pt *.valid.txt valid.log /content/drive/MyDrive/deepalgpro_ablation/
!ls -lh /content/drive/MyDrive/deepalgpro_ablation


ValueError: mount failed

In [ ]:
!python main.py -i data/all.train.fasta --epochs 120 --lr 0.0001 -b 72 --mode train --model ATTnet | tail -n 3


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/deepalgpro_ablation
!cp *.1.pt *.valid.txt valid.log /content/drive/MyDrive/deepalgpro_ablation/
!ls -lh /content/drive/MyDrive/deepalgpro_ablation


In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from Bio import SeqIO
from explore_model import encode, codeadict

seqs, labels = [], []
for rec in SeqIO.parse('data/all.test.fasta', 'fasta'):
    s = str(rec.seq).upper()
    if set(s) <= set(codeadict):
        seqs.append(s)
        labels.append(1 if rec.id.startswith('allergen') else 0)
labels = np.array(labels)
x = torch.tensor([encode(s) for s in seqs])
print(x.shape)


In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, auc

def scores(net, x, batch=64):
    net.eval()
    with torch.no_grad():
        return torch.cat([net(b) for b in torch.split(x, batch)]).squeeze(1).numpy()


In [ ]:
import os
print('x:', x.shape, '| labels:', labels.shape)        # test set loaded? e.g. (1420, 1000) and (1420,)
for f in ['model.pt', 'convNet.1.pt', 'ATTnet.1.pt']:
    print(f, 'OK' if os.path.exists(f) else 'MISSING')


In [ ]:
import model as models
import matplotlib.pyplot as plt

# name in legend: (class in model.py, weights file, colour as in the paper)
nets = {'Without MHSA': ('convNet', 'convNet.1.pt', 'tab:orange'),
        'Without CNN':  ('ATTnet', 'ATTnet.1.pt', 'tab:blue'),
        'DeepAlgPro':   ('convATTnet', 'model.pt', 'tab:red')}

fig, (a, b) = plt.subplots(1, 2, figsize=(12, 5.5))
for name, (cls, path, color) in nets.items():
    net = getattr(models, cls)()                                       # build the empty architecture
    net.load_state_dict(torch.load(path, map_location='cpu'))          # fill in the trained weights
    p = scores(net, x)                                                 # one probability per test protein
    fpr, tpr, _ = roc_curve(labels, p)
    prec, rec, _ = precision_recall_curve(labels, p)
    a.plot(fpr, tpr, c=color, label=f'{name} (AUC = {auc(fpr, tpr):.3f})')
    b.plot(rec, prec, c=color, label=f'{name} (AUC = {auc(rec, prec):.3f})')
    print(name, 'done')

a.plot([0, 1], [0, 1], '--', c='gray', lw=0.8)       # random guessing
b.plot([0, 1], [1, 0], '--', c='gray', lw=0.8)       # the paper draws this diagonal too
a.set(xlabel='False positive rate', ylabel='True positive rate', xlim=(0, 1), ylim=(0, 1.02))
b.set(xlabel='Recall', ylabel='Precision', xlim=(0, 1), ylim=(0, 1.02))
a.set_title('A', loc='left'); b.set_title('B', loc='left')
a.legend(loc='lower center'); b.legend(loc='lower center')
fig.tight_layout()
fig.savefig('ablation.png', dpi=300)
plt.show()


In [ ]:
!pip install -q umap-learn biopython


In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
import torch.nn.functional as F
from Bio import SeqIO
from explore_model import encode, codeadict, load_model

model, _ = load_model('model.pt')                 # fresh model = no leftover hooks

ids, seqs, labels = [], [], []
for path in ['data/all.test.fasta']:              # try ['data/all.train.fasta', 'data/all.test.fasta'] later
    for rec in SeqIO.parse(path, 'fasta'):
        s = str(rec.seq).upper()
        if set(s) <= set(codeadict):
            ids.append(rec.id); seqs.append(s)
            labels.append(1 if rec.id.startswith('allergen') else 0)

labels = np.array(labels)
x = torch.tensor([encode(s) for s in seqs])
print(x.shape, 'allergen fraction:', labels.mean())


In [ ]:
feats = {'input': [], 'conv': [], 'attn': [], 'fc': []}
hooks = [
    model.maxpool.register_forward_hook(lambda m, i, o: feats['conv'].append(i[0].flatten(1))),  # conv + ReLU output
    model.fc1.register_forward_hook(lambda m, i, o: feats['attn'].append(i[0])),                 # attention output (already flat)
    model.fc1.register_forward_hook(lambda m, i, o: feats['fc'].append(o)),                      # fully connected output
]
with torch.no_grad():
    for b in torch.split(x, 64):
        feats['input'].append(F.one_hot(b, 21).float().flatten(1))
        model(b)
for h in hooks:
    h.remove()

feats = {k: torch.cat(v).numpy() for k, v in feats.items()}
for k, v in feats.items():
    print(k, v.shape)


In [ ]:
from sklearn.decomposition import PCA
import umap

emb = {}
for k, v in feats.items():
    v_red = PCA(n_components=50, random_state=0).fit_transform(v) if v.shape[1] > 50 else v
    emb[k] = umap.UMAP(n_components=2, random_state=0).fit_transform(v_red)
    print(k, 'done')


In [ ]:
titles = {'input': 'A', 'conv': 'B', 'attn': 'C', 'fc': 'D'}

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, (k, e) in zip(axes.flat, emb.items()):
    # non-allergens first, so red allergens are drawn on top
    for lab, name, color in [(0, 'non-allergen', 'darkturquoise'), (1, 'allergen', 'indianred')]:
        m = labels == lab
        ax.scatter(e[m, 0], e[m, 1], s=2, c=color, label=name)
    ax.set_title(titles[k], loc='left', fontsize=14)
    ax.set_xlabel('Dimension1'); ax.set_ylabel('Dimension2')
    ax.grid(color='gray', alpha=0.6)
    ax.legend(loc='upper right', markerscale=3)
fig.tight_layout()
fig.savefig('umap_layers.png', dpi=300)
plt.show()


In [ ]:
from sklearn.metrics import silhouette_score
for k, e in emb.items():
    print(k, round(silhouette_score(e, labels), 3))


## Grad-Cam

In [ ]:
from Bio import SeqIO
targets = {'Gad m 1': 'GDGKIGVDEFGAMIKA', 'Fel d 1': 'FAVANGNELLLDLS', 'Ara h 1': 'EQEERGQRRW'}
for path in ['data/all.train.fasta', 'data/all.test.fasta']:
    for rec in SeqIO.parse(path, 'fasta'):
        for name, ep in targets.items():
            if ep in str(rec.seq):
                print(name, path, rec.id, len(rec.seq), 'epitope at', str(rec.seq).find(ep) + 1)


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from gradcam import gradcam, to_residues
from explore_model import load_model

model, logits = load_model('model.pt')

def cam_track(seq):
    _, cams, _, _ = gradcam(model, logits, seq, mode='gradcam')    # standard Grad-CAM, conv layer
    s = np.maximum(to_residues(cams['conv'], len(seq)), 0)         # Grad-CAM keeps only positive evidence (ReLU)
    return s / s.max()                                              # scale to 0–1, like the paper's colorbar


In [ ]:
from Bio import SeqIO
test = {rec.id: str(rec.seq) for rec in SeqIO.parse('data/all.test.fasta', 'fasta')}
gad_seq  = test['allergen_473']
feld_seq = test['allergen_48']
arah_seq = test['allergen_134']
print(len(gad_seq), len(feld_seq), len(arah_seq))


In [ ]:
def plot_track(ax, seq, epitopes, title):
    s = cam_track(seq)
    L = len(seq)
    im = ax.imshow(s[None, :], aspect='auto', cmap='viridis', vmin=0, vmax=1,
                   extent=(0.5, L + 0.5, 0, 1), interpolation='nearest')
    for ep in epitopes:
        i = seq.find(ep)
        if i < 0:
            print('epitope not found:', ep); continue
        start, end = i + 1, i + len(ep)                               # 1-based positions
        ax.add_patch(Rectangle((start - 0.5, 0.01), len(ep), 0.98, fill=False, ec='black', lw=1.2))
        mid = (start + end) / 2
        ax.annotate(ep, xy=(mid, 0), xytext=(mid, -0.55), ha='center', color='red', fontsize=10,
                    arrowprops=dict(arrowstyle='->'), annotation_clip=False)
        ax.text(start, -0.25, str(start), ha='center', fontsize=8, clip_on=False)
        ax.text(end, -0.25, str(end), ha='center', fontsize=8, clip_on=False)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, loc='left', fontsize=13)
    return im

proteins = {
    'F': (gad_seq,  ['GDGKIGVDEFGAMIKA']),
    'G': (feld_seq, ['FAVANGNELLLDLS']),
    'H': (arah_seq, ['SATHAKSSPYQKKT', 'QEPDDLKQKA', 'EQEERGQRRW']),
}
fig, axes = plt.subplots(3, 1, figsize=(12, 7), gridspec_kw={'hspace': 1.2})
for ax, (panel, (seq, eps)) in zip(axes, proteins.items()):
    im = plot_track(ax, seq, eps, panel)
    fig.colorbar(im, ax=ax, fraction=0.02, pad=0.01)
fig.savefig('gradcam_epitopes.png', dpi=300, bbox_inches='tight')
plt.show()


## Figure 4 - interpretability of the convolutional layer

Reproduces He et al. 2023, Figure 4, using `figure4.py`:

* **F-H** Grad-CAM tracks for Gad m 1, Fel d 1 (chain 2) and Ara h 1, with the
  published linear IgE epitopes marked below each axis.
* **A-E** motifs from high-Grad-CAM regions compared against motifs from known
  epitopes (MEME -> Tomtom), drawn as stacked sequence logos.

Paper settings, from *Finding key regions*: windows are +-7 AA around every
residue scoring above **80% of the per-protein max**, MEME with `-nmotifs 200`
on the Grad-CAM windows and `-evt 0.05` on the epitopes, then Tomtom to compare.

In [ ]:
%cd /content/deep-alg-pro
!git pull origin lab
!ls -l figure4.py gradcam.py

### Panels F-H

`--mode gradcam` is the paper-faithful setting (standard Grad-CAM: one weight per
filter, averaged over all 985 windows). The script also prints the quantitative
check - peak residue, mean weight inside vs outside the epitope, and the share of
above-threshold residues that fall inside an epitope.

In [ ]:
!python figure4.py panels --weights model.pt --fasta data/all.test.fasta -o figure4fh.png

from IPython.display import Image, display
display(Image('figure4fh.png'))

In [ ]:
# position-aware variant: fc1 has separate weights per window, so averaging the
# gradient over windows (standard Grad-CAM) discards position information
!python figure4.py panels --mode gradxact -o figure4fh.gradxact.png
display(Image('figure4fh.gradxact.png'))

### Panels A-E, step 1: the Grad-CAM windows

One 15-mer per residue scoring above 0.8 x max, across every test-set allergen the
model predicts correctly. Expect roughly 20k windows from the 710 test positives.

In [ ]:
!python figure4.py extract --fasta data/all.test.fasta -o gradcam_windows.fa
!head -4 gradcam_windows.fa && echo "..." && grep -c '^>' gradcam_windows.fa

### Step 2: the epitope set

`data/epitope_table_export_1793356730.csv` is an **IEDB** export (iedb.org ->
Epitope Search -> Linear Peptide + B cell assays -> Export CSV): 6269 linear
peptides, mostly from peanut, soy, wheat, cow milk, egg, latex, shrimp and dust
mite. 5313 survive the length and `BJOUXZ` filters below.

The export has **two header rows**, and the peptide sits in `Epitope | Name` -
there is no column called "Sequence".

The paper also used AllerBase and IgPred, which have no bulk download; using IEDB
alone is a deviation worth stating in the report.

In [ ]:
import pandas as pd

IEDB_CSV = 'data/epitope_table_export_1793356730.csv'
STD = set('ACDEFGHIKLMNPQRSTVWY')

df = pd.read_csv(IEDB_CSV, low_memory=False, header=[0, 1])       # two header rows
df.columns = ['|'.join(str(x) for x in c) for c in df.columns]

linear = df[df['Epitope|Object Type'] == 'Linear peptide']        # drop discontinuous epitopes
peps = sorted({str(s).strip().upper() for s in linear['Epitope|Name'].dropna()})
peps = [p for p in peps if 8 <= len(p) <= 40 and set(p) <= STD]   # same BJOUXZ rule as the dataset

with open('epitopes.fa', 'w', newline='\n') as fh:
    for i, p in enumerate(peps):
        fh.write(f'>epitope_{i}\n{p}\n')
print(len(peps), 'epitopes written to epitopes.fa')

### Step 3: MEME and Tomtom

MEME Suite is Linux-only, which is why this part runs in Colab rather than locally.

`-nmotifs 200` on ~20k sequences is slow (tens of minutes). If it takes too long,
subsample `gradcam_windows.fa` with a fixed seed and say so in the report.

In [ ]:
!apt-get -qq install -y meme > /dev/null
!meme -version && tomtom -version

In [ ]:
%%time
# -protein is required (MEME defaults to DNA); -mod zoops = zero or one occurrence per sequence
!meme gradcam_windows.fa -protein -oc meme_gradcam  -nmotifs 200 -mod zoops -minw 6 -maxw 15
!meme epitopes.fa        -protein -oc meme_epitopes -evt 0.05    -mod zoops

In [ ]:
# query = epitope motifs, target = Grad-CAM motifs, so the paper's panel order
# (epitope on top, Grad-CAM below) comes out right
!tomtom -oc tomtom_out -thresh 0.001 -evalue meme_epitopes/meme.txt meme_gradcam/meme.txt
!head -3 tomtom_out/tomtom.tsv

### Step 4: the A-E panels

In [ ]:
!python figure4.py logos --top 5 --pmax 1e-3 -o figure4ae.png
display(Image('figure4ae.png'))

In [ ]:
import pandas as pd

t = pd.read_csv('tomtom_out/tomtom.tsv', sep='\t', comment='#')
t.columns = [c.strip() for c in t.columns]
sig = t[t['p-value'] < 1e-3].sort_values('p-value')
print(f'{len(sig)} matched pairs with p < 1e-3   (the paper reports 11)')
print(sig[['Query_ID', 'Target_ID', 'p-value']].head(10).to_string(index=False))

In [ ]:
# keep the figures across runtime restarts
!mkdir -p /content/drive/MyDrive/deepalgpro_ablation
!cp -f figure4fh.png figure4fh.gradxact.png figure4ae.png gradcam_windows.fa \
      /content/drive/MyDrive/deepalgpro_ablation/ 2>/dev/null
!ls -lh /content/drive/MyDrive/deepalgpro_ablation